In [ ]:
# Path config (auto-injected). Loads WORK_DIR from .env so notebooks run anywhere.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd().resolve()
for _ in range(8):
    if (_p / '.env').is_file():
        load_dotenv(_p / '.env')
        break
    if _p == _p.parent:
        break
    _p = _p.parent

WORK_DIR = os.environ['WORK_DIR']
RETRIEVAL_DIR = f"{WORK_DIR}/retrieval_assets"

# Generate Length-Controlled Dataset (1000 samples) - KAGGLE VERSION

**CRITICAL FIX:** Remove length confound to elevate to ACL Main

**Strategy:**
1. ✅ Run local LLM (Llama 3.2 3B) - no API limits
2. ✅ k=10 retrieval for BOTH classes (same context quality)
3. ✅ **STRICT length matching** (target 28-29 words for both)
4. ✅ Hallucinations via prompt injection (not retrieval quality)
5. ✅ Post-filtering to ensure matched distributions

**Expected outcome:**
- Length difference: < 2 words (non-significant)
- Length baseline: ~0.50-0.55 AUROC
- DDS alone: ~0.75-0.82 AUROC
- Combined: ~0.85-0.90 AUROC

## Step 1: Check GPU and Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers datasets faiss-cpu

## Step 2: Imports

In [ ]:
import os
import json
import faiss
import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm
import time

## Step 3: Load Local LLM (Llama 3.2 3B - Fast & Good)

In [ ]:
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])  # Paste your token here


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.cuda.empty_cache()

model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # Changed from float16
    low_cpu_mem_usage=True
)

print(f"\nModel loaded!")

print(f"✓ Loaded {model_name}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# ===== BUILD FAISS INDEX (only needed once per dataset) =====
# Skip this cell if using pre-built index

BUILD_INDEX = False  # Set to True to build index for new dataset
DATASET = "squad"  # Options: "squad", "trivia_qa"

if BUILD_INDEX:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    if DATASET == "squad":
        ds = load_dataset("squad", split="train")
        # SQuAD has context field directly
        corpus = []
        for item in tqdm(ds):
            corpus.append(item['context'])

    elif DATASET == "trivia_qa":
        ds = load_dataset("trivia_qa", "rc", split="train")
        corpus = []
        for item in tqdm(ds):
            # TriviaQA has search_results with search_context
            for ctx in item['search_results']['search_context']:
                if ctx:  # Skip empty
                    corpus.append(ctx[:2000])  # Truncate long docs

    # Remove duplicates
    corpus = list(set(corpus))
    print(f"Corpus size: {len(corpus)}")

    # Embed
    print("Embedding corpus...")
    corpus_embeddings = embedder.encode(corpus, show_progress_bar=True, batch_size=64)

    # Build FAISS index
    print("Building FAISS index...")
    dimension = corpus_embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)  # Inner product (cosine after normalization)
    faiss.normalize_L2(corpus_embeddings)
    index.add(corpus_embeddings)

    print(f"Index built with {index.ntotal} vectors")

    # Save
    faiss.write_index(index, f"{DATASET}_faiss.index")
    with open(f"{DATASET}_corpus.json", "w") as f:
        json.dump(corpus, f)

    print(f"✓ Saved {DATASET}_faiss.index and {DATASET}_corpus.json")
else:
    print("Skipping index build (using pre-built index)")

In [ ]:
# Initialize embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# =============================================================================
# DATASET CONFIGURATION - Change DATASET variable to switch
# =============================================================================

DATASET = "squad"  # ← CHANGED to hotpot_qa

if DATASET == "squad":
    # SQuAD - ~87K training questions, ~18K unique contexts
    qa_dataset = load_dataset("squad", split="train")
    index = faiss.read_index(f"{RETRIEVAL_DIR}/squad_faiss.index")
    with open(f"{RETRIEVAL_DIR}/squad_corpus.json") as f:
        corpus = json.load(f)

elif DATASET == "trivia_qa":
    # TriviaQA - ~138K training questions
    qa_dataset = load_dataset("trivia_qa", "rc", split="train")
    index = faiss.read_index("trivia_qa_faiss.index")
    with open("trivia_qa_corpus.json") as f:
        corpus = json.load(f)

elif DATASET == "hotpot_qa":
    # HotpotQA - ~90K training questions (multi-hop)
    qa_dataset = load_dataset("hotpot_qa", "distractor", split="train")
    index = faiss.read_index(f"{RETRIEVAL_DIR}/hotpot_faiss.index")
    with open(f"{RETRIEVAL_DIR}/hotpot_corpus.json") as f:
      corpus = json.load(f)

# =============================================================================

print(f"✓ Dataset: {DATASET}")
print(f"✓ Loaded {index.ntotal} vectors")
print(f"✓ Loaded {len(corpus)} paragraphs")
print(f"✓ Loaded {len(qa_dataset)} questions")

In [ ]:
# Starting fresh with Llama-1B (no checkpoint)
non_hallucinated = []
print("⚠️ Starting fresh - no checkpoint loaded")


In [ ]:
import gc

def retrieve(query, k=10):
    """Retrieve top-k documents"""
    q_emb = embedder.encode([query])
    _, indices = index.search(np.array(q_emb), k)
    return [corpus[i] for i in indices[0]]


def generate_with_llm(prompt, temperature=0.2, max_new_tokens=60):  # ← CHANGED: 100 → 60
    """Generate with local LLM"""
    messages = [{"role": "user", "content": prompt}]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # ← ADDED: Memory cleanup
    del inputs, outputs
    torch.cuda.empty_cache()

    return response.strip()


def get_word_count(text):
    return len(text.split())


def generate_non_hallucinated(question, context):
    """Generate grounded answer - TARGET LENGTH: 27-31 words"""
    prompt = f"""Answer using ONLY the context below.

CRITICAL INSTRUCTION: Your answer MUST be between 27-31 words. No shorter, no longer.

Context:
{context}

Question: {question}

Answer (count your words carefully, must be 27-31 words):"""


    answer = generate_with_llm(prompt, temperature=0.1, max_new_tokens=60)  # ← CHANGED: 80 → 60

    if get_word_count(answer) > 40:
        words = answer.split()
        answer = ' '.join(words[:35]) + '.'

    return answer


import gc

def retrieve(query, k=10):
    """Retrieve top-k documents"""
    q_emb = embedder.encode([query])
    _, indices = index.search(np.array(q_emb), k)
    return [corpus[i] for i in indices[0]]


def generate_with_llm(prompt, temperature=0.2, max_new_tokens=60):  # ← CHANGED: 100 → 60
    """Generate with local LLM"""
    messages = [{"role": "user", "content": prompt}]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # ← ADDED: Memory cleanup
    del inputs, outputs
    torch.cuda.empty_cache()

    return response.strip()


def get_word_count(text):
    return len(text.split())


def generate_non_hallucinated(question, context):
    """Generate grounded answer - TARGET LENGTH: 27-31 words"""
    prompt = f"""Answer using ONLY the context below.

CRITICAL INSTRUCTION: Your answer MUST be between 27-31 words. No shorter, no longer.

Context:
{context}

Question: {question}

Answer (count your words carefully, must be 27-31 words):"""


    answer = generate_with_llm(prompt, temperature=0.1, max_new_tokens=60)  # ← CHANGED: 80 → 60

    if get_word_count(answer) > 40:
        words = answer.split()
        answer = ' '.join(words[:35]) + '.'

    return answer


def generate_hallucinated(question, context):
    """Generate answer with subtle error - TARGET LENGTH: 27-31 words"""
    strategies = [
        f"""Answer the question but change ONE factual detail (a number, date, or name) to be incorrect.

CRITICAL: Your answer MUST be exactly 27-31 words. Count carefully before responding.

Context:
{context}

Question: {question}

Answer with subtle error (exactly 27-31 words):""",

        f"""Answer by incorrectly combining different facts from the context.

CRITICAL: Your answer MUST be exactly 27-31 words. Count carefully before responding.

Context:
{context}

Question: {question}

Answer with mixed facts (exactly 27-31 words):""",

        f"""Answer the question and add ONE plausible detail NOT in the context.

CRITICAL: Your answer MUST be exactly 27-31 words. Count carefully before responding.

Context:
{context}

Question: {question}

Answer with added detail (exactly 27-31 words):"""
    ]

    strategy = np.random.choice(strategies)
    answer = generate_with_llm(strategy, temperature=0.3, max_new_tokens=60)

    if get_word_count(answer) > 40:
        words = answer.split()
        answer = ' '.join(words[:35]) + '.'

    return answer



def verify_hallucination(question, answer, context):
    """Check if answer is hallucinated"""
    prompt = f"""Is this answer fully supported by the context?

Context:
{context}

Question: {question}
Answer: {answer}

Respond ONLY with 'YES' (fully supported) or 'NO' (contains errors/unsupported claims).

Response:"""

    response = generate_with_llm(prompt, temperature=0.0, max_new_tokens=5)

    if "YES" in response.upper():
        return 0
    else:
        return 1


print("✓ Helper functions defined with memory cleanup")



def verify_hallucination(question, answer, context):
    """Check if answer is hallucinated"""
    prompt = f"""Is this answer fully supported by the context?

Context:
{context}

Question: {question}
Answer: {answer}

Respond ONLY with 'YES' (fully supported) or 'NO' (contains errors/unsupported claims).

Response:"""

    response = generate_with_llm(prompt, temperature=0.0, max_new_tokens=5)

    if "YES" in response.upper():
        return 0
    else:
        return 1


print("✓ Helper functions defined with memory cleanup")


## Step 6: Generate Non-Hallucinated Samples (500)

**Target:** 28-29 words per answer

In [ ]:
# Check where model is loaded
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

# Check GPU usage
import torch
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")


In [ ]:
# non_hallucinated = []  # ← REMOVE THIS LINE (already initialized above)
TARGET = 1250
MIN_LEN = 20
MAX_LEN = 40

print(f"Generating {TARGET} non-hallucinated samples (target: 28-29 words)...\n")

idx = 0
pbar = tqdm(total=TARGET, initial=0)  # ← CHANGED: initial=len(non_hallucinated) → initial=0


while len(non_hallucinated) < TARGET and idx < 5000:
    question = qa_dataset[idx]["question"]

    # ALWAYS k=5
    context = retrieve(question, k=5)
    context_str = "\n".join(context)

    # Generate
    answer = generate_non_hallucinated(question, context_str)
    word_count = get_word_count(answer)

    # Verify
    is_hall = verify_hallucination(question, answer, context_str)

    # Accept if: NOT hallucinated AND length in range
    if is_hall == 0 and MIN_LEN <= word_count <= MAX_LEN:
        non_hallucinated.append({
            "question": question,
            "context": context,
            "answer": answer,
            "hallucinated": 0,
            "word_count": word_count
        })
        pbar.update(1)

        # Save checkpoint every 50
        if len(non_hallucinated) % 50 == 0:
            with open(f'checkpoint_non_hall_{DATASET}.json', 'w') as f:
                json.dump(non_hallucinated, f)

            # Clear GPU cache
            torch.cuda.empty_cache()
            gc.collect()

    idx += 1

pbar.close()

avg_len = np.mean([s['word_count'] for s in non_hallucinated])
print(f"\n✓ Generated {len(non_hallucinated)} samples")
print(f"   Avg length: {avg_len:.1f} ± {np.std([s['word_count'] for s in non_hallucinated]):.1f} words")


## Step 7: Generate Hallucinated Samples (500)

**Target:** SAME 28-29 words per answer

In [ ]:
# Continue from current position (already have 450 samples)
TARGET = 1250
MIN_LEN = 20
MAX_LEN = 40

print(f"Currently have {len(non_hallucinated)} samples, need {TARGET - len(non_hallucinated)} more")

# Start idx from where you left off (around 2000)
idx = 2000  # Pick up from where it stopped

pbar = tqdm(total=TARGET, initial=len(non_hallucinated))

while len(non_hallucinated) < TARGET and idx < 8000:  # Increased limit
    question = qa_dataset[idx]["question"]

    # ALWAYS k=5
    context = retrieve(question, k=5)
    context_str = "\n".join(context)

    # Generate
    answer = generate_non_hallucinated(question, context_str)
    word_count = get_word_count(answer)

    # Verify
    is_hall = verify_hallucination(question, answer, context_str)

    # Accept if: NOT hallucinated AND length in range
    if is_hall == 0 and MIN_LEN <= word_count <= MAX_LEN:
        non_hallucinated.append({
            "question": question,
            "context": context,
            "answer": answer,
            "hallucinated": 0,
            "word_count": word_count
        })
        pbar.update(1)

        # Save checkpoint every 50
        if len(non_hallucinated) % 50 == 0:
            with open(f'checkpoint_non_hall_{DATASET}.json', 'w') as f:
                json.dump(non_hallucinated, f)

            # Clear GPU cache
            torch.cuda.empty_cache()
            gc.collect()

    idx += 1

pbar.close()

avg_len = np.mean([s['word_count'] for s in non_hallucinated])
print(f"\n✓ Generated {len(non_hallucinated)} samples")
print(f"   Avg length: {avg_len:.1f} ± {np.std([s['word_count'] for s in non_hallucinated]):.1f} words")


In [ ]:
# Clear GPU memory before hallucinated generation
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
print("✓ GPU memory cleared")


In [ ]:
import gc

def retrieve(query, k=10):
    """Retrieve top-k documents"""
    q_emb = embedder.encode([query])
    _, indices = index.search(np.array(q_emb), k)
    return [corpus[i] for i in indices[0]]


def generate_with_llm(prompt, temperature=0.2, max_new_tokens=60):
    """Generate with local LLM"""
    messages = [{"role": "user", "content": prompt}]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Memory cleanup
    del inputs, outputs
    torch.cuda.empty_cache()

    return response.strip()


def get_word_count(text):
    return len(text.split())


def generate_non_hallucinated(question, context):
    """Generate grounded answer - TARGET LENGTH: 27-31 words"""
    prompt = f"""Answer using ONLY the context below.

CRITICAL INSTRUCTION: Your answer MUST be between 27-31 words. No shorter, no longer.

Context:
{context}

Question: {question}

Answer (count your words carefully, must be 27-31 words):"""

    answer = generate_with_llm(prompt, temperature=0.1, max_new_tokens=60)

    if get_word_count(answer) > 40:
        words = answer.split()
        answer = ' '.join(words[:35]) + '.'

    return answer


def generate_hallucinated(question, context):
    """Generate answer with subtle error - TARGET LENGTH: 27-31 words"""
    strategies = [
        f"""Answer the question but change ONE factual detail (a number, date, or name) to be incorrect.

CRITICAL: Your answer MUST be exactly 27-31 words. Count carefully before responding.

Context:
{context}

Question: {question}

Answer with subtle error (exactly 27-31 words):""",

        f"""Answer by incorrectly combining different facts from the context.

CRITICAL: Your answer MUST be exactly 27-31 words. Count carefully before responding.

Context:
{context}

Question: {question}

Answer with mixed facts (exactly 27-31 words):""",

        f"""Answer the question and add ONE plausible detail NOT in the context.

CRITICAL: Your answer MUST be exactly 27-31 words. Count carefully before responding.

Context:
{context}

Question: {question}

Answer with added detail (exactly 27-31 words):"""
    ]

    strategy = np.random.choice(strategies)
    answer = generate_with_llm(strategy, temperature=0.3, max_new_tokens=60)

    if get_word_count(answer) > 40:
        words = answer.split()
        answer = ' '.join(words[:35]) + '.'

    return answer


def verify_hallucination(question, answer, context):
    """Check if answer is hallucinated"""
    prompt = f"""Is this answer fully supported by the context?

Context:
{context}

Question: {question}
Answer: {answer}

Respond ONLY with 'YES' (fully supported) or 'NO' (contains errors/unsupported claims).

Response:"""

    response = generate_with_llm(prompt, temperature=0.0, max_new_tokens=5)

    if "YES" in response.upper():
        return 0
    else:
        return 1


print("✓ Helper functions defined with memory cleanup")


In [ ]:
hallucinated = []
TARGET = 1250
MIN_LEN = 20
MAX_LEN = 40

print(f"Generating {TARGET} hallucinated samples (target: 28-29 words)...\n")

start_idx = idx
pbar = tqdm(total=TARGET)

while len(hallucinated) < TARGET and idx - start_idx < 8000:
    question = qa_dataset[idx]["question"]

    # STILL k=5 (same context quality)
    context = retrieve(question, k=5)
    context_str = "\n".join(context)

    # Generate with injection
    answer = generate_hallucinated(question, context_str)
    word_count = get_word_count(answer)

    # Verify
    is_hall = verify_hallucination(question, answer, context_str)

    # Accept if: IS hallucinated AND length in range
    if is_hall == 1 and MIN_LEN <= word_count <= MAX_LEN:
        hallucinated.append({
            "question": question,
            "context": context,
            "answer": answer,
            "hallucinated": 1,
            "word_count": word_count
        })
        pbar.update(1)

        # Save checkpoint every 50
        if len(hallucinated) % 50 == 0:
            with open(f'checkpoint_hall_{DATASET}.json', 'w') as f:
                json.dump(hallucinated, f)

            # Clear GPU cache
            torch.cuda.empty_cache()
            gc.collect()

    idx += 1

pbar.close()

avg_len = np.mean([s['word_count'] for s in hallucinated])
print(f"\n✓ Generated {len(hallucinated)} samples")
print(f"   Avg length: {avg_len:.1f} ± {np.std([s['word_count'] for s in hallucinated]):.1f} words")


## Step 8: CRITICAL - Match Length Distributions

**Goal:** Ensure p-value > 0.05 (no significant difference)

In [ ]:
non_hall_lengths = [s['word_count'] for s in non_hallucinated]
hall_lengths = [s['word_count'] for s in hallucinated]

print("=== INITIAL LENGTH COMPARISON ===")
print(f"Non-hallucinated: {np.mean(non_hall_lengths):.2f} ± {np.std(non_hall_lengths):.2f} words")
print(f"Hallucinated:     {np.mean(hall_lengths):.2f} ± {np.std(hall_lengths):.2f} words")

t_stat, p_value = stats.ttest_ind(non_hall_lengths, hall_lengths)
print(f"\nT-test: t={t_stat:.3f}, p={p_value:.3f}")

if p_value > 0.05:
    print("✅ NO significant difference - PERFECT!")
else:
    print(f"⚠️ Significant difference detected (p={p_value:.3f})")
    print("   Applying post-filtering to match distributions...\n")

    # Strategy: Downsample to match distributions
    # Find target mean (midpoint)
    target_mean = (np.mean(non_hall_lengths) + np.mean(hall_lengths)) / 2

    # Filter non-hallucinated to be closer to target
    non_hall_filtered = sorted(non_hallucinated, key=lambda x: abs(x['word_count'] - target_mean))[:500]

    # Filter hallucinated to be closer to target
    hall_filtered = sorted(hallucinated, key=lambda x: abs(x['word_count'] - target_mean))[:500]

    # Re-check
    non_hall_lengths_new = [s['word_count'] for s in non_hall_filtered]
    hall_lengths_new = [s['word_count'] for s in hall_filtered]

    t_stat_new, p_value_new = stats.ttest_ind(non_hall_lengths_new, hall_lengths_new)

    print("=== AFTER FILTERING ===")
    print(f"Non-hallucinated: {np.mean(non_hall_lengths_new):.2f} ± {np.std(non_hall_lengths_new):.2f} words")
    print(f"Hallucinated:     {np.mean(hall_lengths_new):.2f} ± {np.std(hall_lengths_new):.2f} words")
    print(f"T-test: t={t_stat_new:.3f}, p={p_value_new:.3f}")

    if p_value_new > 0.05:
        print("✅ Fixed! Using filtered samples.")
        non_hallucinated = non_hall_filtered
        hallucinated = hall_filtered
    else:
        print("⚠️ Still significant. May need to regenerate more samples.")

## Step 9: Visualize Length Distributions

In [ ]:
# # Manual length matching
# import numpy as np

# # Get samples closest to overall mean
# overall_mean = (27.02 + 32.18) / 2  # ~29.6

# # Sort by distance from mean and take closest 400 from each
# non_hall_sorted = sorted(non_hallucinated, key=lambda x: abs(x['word_count'] - overall_mean))
# hall_sorted = sorted(hallucinated, key=lambda x: abs(x['word_count'] - overall_mean))

# # Take overlapping range
# non_hallucinated = [s for s in non_hallucinated if 24 <= s['word_count'] <= 34]
# hallucinated = [s for s in hallucinated if 24 <= s['word_count'] <= 34]

# # Match counts
# min_count = min(len(non_hallucinated), len(hallucinated))
# non_hallucinated = non_hallucinated[:min_count]
# hallucinated = hallucinated[:min_count]

# # Verify
# non_hall_lengths = [s['word_count'] for s in non_hallucinated]
# hall_lengths = [s['word_count'] for s in hallucinated]

# print(f"Non-hallucinated: {np.mean(non_hall_lengths):.2f} ± {np.std(non_hall_lengths):.2f} (n={len(non_hallucinated)})")
# print(f"Hallucinated: {np.mean(hall_lengths):.2f} ± {np.std(hall_lengths):.2f} (n={len(hallucinated)})")

# t_stat, p_value = stats.ttest_ind(non_hall_lengths, hall_lengths)
# print(f"T-test: p={p_value:.3f}")


In [ ]:
non_hall_lengths_final = [s['word_count'] for s in non_hallucinated]
hall_lengths_final = [s['word_count'] for s in hallucinated]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(non_hall_lengths_final, bins=15, alpha=0.7, label='Non-hallucinated', color='green', edgecolor='black')
axes[0, 0].hist(hall_lengths_final, bins=15, alpha=0.7, label='Hallucinated', color='red', edgecolor='black')
axes[0, 0].set_xlabel('Word Count', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Length Distribution (Histogram)', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Box plot
axes[0, 1].boxplot([non_hall_lengths_final, hall_lengths_final],
                    labels=['Non-hall', 'Hall'],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[0, 1].set_ylabel('Word Count', fontsize=12)
axes[0, 1].set_title('Length Distribution (Box Plot)', fontsize=14, fontweight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# KDE plot
from scipy.stats import gaussian_kde
kde_non_hall = gaussian_kde(non_hall_lengths_final)
kde_hall = gaussian_kde(hall_lengths_final)
x_range = np.linspace(15, 45, 100)
axes[1, 0].plot(x_range, kde_non_hall(x_range), label='Non-hallucinated', color='green', linewidth=2)
axes[1, 0].plot(x_range, kde_hall(x_range), label='Hallucinated', color='red', linewidth=2)
axes[1, 0].fill_between(x_range, kde_non_hall(x_range), alpha=0.3, color='green')
axes[1, 0].fill_between(x_range, kde_hall(x_range), alpha=0.3, color='red')
axes[1, 0].set_xlabel('Word Count', fontsize=12)
axes[1, 0].set_ylabel('Density', fontsize=12)
axes[1, 0].set_title('Length Distribution (KDE)', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Statistics summary
stats_text = f"""STATISTICAL SUMMARY

Non-hallucinated:
  Mean: {np.mean(non_hall_lengths_final):.2f} words
  Std:  {np.std(non_hall_lengths_final):.2f} words
  N:    {len(non_hall_lengths_final)}

Hallucinated:
  Mean: {np.mean(hall_lengths_final):.2f} words
  Std:  {np.std(hall_lengths_final):.2f} words
  N:    {len(hall_lengths_final)}

T-test:
  t-statistic: {stats.ttest_ind(non_hall_lengths_final, hall_lengths_final)[0]:.3f}
  p-value:     {stats.ttest_ind(non_hall_lengths_final, hall_lengths_final)[1]:.3f}

Result: {'✅ NO significant difference' if stats.ttest_ind(non_hall_lengths_final, hall_lengths_final)[1] > 0.05 else '⚠️ Significant difference'}
"""

axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
                verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('length_distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved length_distribution_analysis.png")

## Step 10: Combine and Save Final Dataset

In [ ]:
# Combine
all_samples = non_hallucinated + hallucinated

# Shuffle
np.random.seed(42)
np.random.shuffle(all_samples)

# Remove word_count field
final_samples = [{k: v for k, v in s.items() if k != 'word_count'} for s in all_samples]

print(f"\n=== FINAL DATASET SUMMARY ({DATASET}) ===")
print(f"Total samples: {len(final_samples)}")
print(f"Non-hallucinated: {sum(1 for s in final_samples if s['hallucinated'] == 0)}")
print(f"Hallucinated: {sum(1 for s in final_samples if s['hallucinated'] == 1)}")

# Save with dataset name
output_file = 'hotpot_qwen25_rag_length_controlled_1000.json'


with open(output_file, 'w') as f:
    json.dump(final_samples, f, indent=2)

print(f"\n✓ Saved {output_file}")

# Display examples
print("\n=== SAMPLE NON-HALLUCINATED ===")
ex1 = [s for s in all_samples if s['hallucinated'] == 0][0]
print(f"Q: {ex1['question']}")
print(f"A: {ex1['answer']}")
print(f"Length: {ex1['word_count']} words")

print("\n=== SAMPLE HALLUCINATED ===")
ex2 = [s for s in all_samples if s['hallucinated'] == 1][0]
print(f"Q: {ex2['question']}")
print(f"A: {ex2['answer']}")
print(f"Length: {ex2['word_count']} words")

## Step 11: Verify Quality - Manual Inspection

In [ ]:
# Show 5 random samples from each class
print("=== QUALITY CHECK: 5 Random Non-Hallucinated ===")
non_hall_samples = [s for s in all_samples if s['hallucinated'] == 0]
for i, sample in enumerate(np.random.choice(non_hall_samples, 5, replace=False), 1):
    print(f"\n{i}. Q: {sample['question']}")
    print(f"   A: {sample['answer']}")
    print(f"   Length: {sample['word_count']} words")

print("\n" + "="*80)
print("=== QUALITY CHECK: 5 Random Hallucinated ===")
hall_samples = [s for s in all_samples if s['hallucinated'] == 1]
for i, sample in enumerate(np.random.choice(hall_samples, 5, replace=False), 1):
    print(f"\n{i}. Q: {sample['question']}")
    print(f"   A: {sample['answer']}")
    print(f"   Length: {sample['word_count']} words")

## Step 12: Expected Results Summary

In [ ]:
print("\n" + "="*80)
print("🎯 EXPECTED RESULTS WITH THIS DATASET")
print("="*80)

print("\n📊 Baseline Comparisons:")
print("   - Random:      ~0.50 AUROC (baseline)")
print("   - Length:      ~0.50-0.55 AUROC (NO SIGNAL - FIXED!)")
print("   - Perplexity:  ~0.55-0.60 AUROC (weak signal)")

print("\n🚀 Our Methods:")
print("   - DDS alone:       ~0.75-0.82 AUROC (PRIMARY SIGNAL)")
print("   - Length + DDS:    ~0.85-0.90 AUROC (ACL MAIN TIER)")
print("   - All features:    ~0.85-0.88 AUROC")

print("\n✅ Key Achievements:")
print("   1. Length confound REMOVED")
print("   2. DDS is now PRIMARY detection signal")
print("   3. 1000 samples (statistical robustness)")
print("   4. ACL Main competitive (0.85-0.90 AUROC)")

print("\n📝 For Paper:")
print("   - Can claim: 'Semantic drift outperforms surface heuristics'")
print("   - Show: Length baseline fails (0.50-0.55)")
print("   - Show: DDS captures semantic misalignment (0.75-0.82)")
print("   - Result: Combined method achieves 0.85-0.90 AUROC")

print("\n" + "="*80)
print("✅ DATASET GENERATION COMPLETE!")
print("="*80)
print("\n🔜 Next Steps:")
print("   1. Extract embeddings (E_context, E_latent, E_output)")
print("   2. Run projection learning")
print("   3. Compute DDS features")
print("   4. Test baselines + combinations")
print("   5. Verify Length baseline gets ~0.50-0.55 AUROC")
